# Can we predict `passed_quiz`?
Each row is one tutoring session; the target is whether the student passed the closing quiz.
Questions: (1) how well can we predict it, (2) is that better than trivial baselines, (3) which signals matter.

Key design choice: the 12,453 sessions come from only 1,800 students (~7 each), so rows from the same student are not independent. Every split below is **grouped by student**; a random row split would let the model see a student's other sessions and overstate performance on new students.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows')
print(df.describe().T)
g = df.groupby('student_key')
print('students:', g.ngroups, '| sessions/student:', g.size().min(), '-', g.size().max())
print('enrol_year / school_code constant within student:', (g[['enrol_year','school_code']].nunique() == 1).all().all())
print('prior_score varies within student (median #distinct values):', g.prior_score.nunique().median())
print('prior_score > 100:', (df.prior_score > 100).sum(), 'rows')
print('base rate passed:', df.passed_quiz.mean().round(3))

Notes: no missing values or duplicates. `prior_score` is described as a *placement test* score, which should be one number per student, but it differs between a student's sessions, so it is a noisy per-session copy. 19 values exceed 100, which looks odd for a score; I leave them as they are and flag them.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
for a, col in zip(ax[:3], ['prior_score', 'session_minutes', 'hints_used']):
    q = pd.qcut(df[col], 10, duplicates='drop')
    r = df.groupby(q, observed=True).passed_quiz.mean()
    a.plot(range(len(r)), r.values, 'o-'); a.set_title(f'pass rate by {col} decile'); a.set_ylim(0, 0.8)
sr = g.passed_quiz.mean()
ax[3].hist(sr, bins=20); ax[3].set_title('per-student pass rate')
plt.tight_layout(); plt.show()
p0 = df.passed_quiz.mean()
print('SD of per-student pass rates: observed %.3f vs %.3f expected if every student had the same pass rate' % (sr.std(), np.sqrt(p0*(1-p0)/g.size().mean())))

Session length and hints show monotone relationships with passing; `prior_score` shows little. Per-student pass rates are spread more widely than a common rate would give for ~7 sessions each (see the printed comparison), which points to a student-level effect we can't observe directly for a new student.

## 2. Models (5-fold CV, grouped by student)

In [ ]:
feats = ['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']
X, y, grp = df[feats], df.passed_quiz, df.student_key
cv = GroupKFold(n_splits=5)
mm = ['session_minutes', 'hints_used']
models = {
    'baseline (base rate)':    (feats, DummyClassifier(strategy='prior')),
    'logistic: minutes+hints': (mm,    make_pipeline(StandardScaler(), LogisticRegression())),
    'logistic: all':           (feats, make_pipeline(StandardScaler(), LogisticRegression())),
    'grad. boosting: all':     (feats, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
rows, preds = [], {}
for n, (cols, est) in models.items():
    p = cross_val_predict(est, X[cols], y, groups=grp, cv=cv, method='predict_proba')[:, 1]
    preds[n] = p
    aucs = [roc_auc_score(y.iloc[te], p[te]) for _, te in cv.split(X, y, grp)]
    rows.append((n, np.mean(aucs), np.std(aucs), log_loss(y, p), accuracy_score(y, p > 0.5)))
res = pd.DataFrame(rows, columns=['model', 'AUC', 'AUC fold SD', 'log loss', 'accuracy']).set_index('model')
print('majority-class accuracy:', round(max(y.mean(), 1 - y.mean()), 3))
res.round(4)

AUC 0.5 is chance; accuracy should be judged against the majority-class rate, not 50%. Differences smaller than the fold SD aren't meaningful.

In [ ]:
lr = models['logistic: all'][1].fit(X, y)
print(pd.Series(lr[-1].coef_[0], index=feats).round(3).rename('std. coefficient (log-odds per SD)'))
best = preds['grad. boosting: all']
bins = pd.qcut(best, 10, labels=False)
cal = pd.DataFrame({'pred': best, 'obs': y}).groupby(bins).mean()
plt.figure(figsize=(4, 4)); plt.plot(cal.pred, cal.obs, 'o-'); plt.plot([0, 1], [0, 1], 'k--', lw=.8)
plt.xlabel('predicted'); plt.ylabel('observed'); plt.title('Calibration (out-of-fold)'); plt.show()

In [ ]:
gb = models['grad. boosting: all'][1].fit(X, y)
pi = permutation_importance(gb, X, y, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=feats).sort_values().plot.barh(title='Permutation importance (AUC drop; in-sample fit)')
plt.show()

## 3. Does knowing the student help?
For a *new* student we know nothing about them; for a returning student we have earlier outcomes. As a rough upper bound, add each student's pass rate on their **other** sessions (leave-one-out). This only applies if those outcomes are known at prediction time, so it is a separate, optimistic scenario.

In [ ]:
s = df.groupby('student_key').passed_quiz.transform('sum'); n = df.groupby('student_key').passed_quiz.transform('count')
df['loo_rate'] = ((s - df.passed_quiz) / (n - 1)).fillna(y.mean())   # excludes the row's own label
f2 = ['session_minutes', 'hints_used', 'loo_rate']
p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), df[f2], y, groups=grp, cv=cv, method='predict_proba')[:, 1]
print('AUC with other-session pass rate: %.4f' % roc_auc_score(y, p))

Caveat: leave-one-out rates can be biased against the row's own label (a known artefact), so treat this as indicative of an upper bound.

## Conclusions

In [ ]:
from IPython.display import Markdown
b = res.loc['grad. boosting: all']; l = res.loc['logistic: minutes+hints']
Markdown(f'''- Grouped-CV AUC is {b.AUC:.3f} (boosting) and {l.AUC:.3f} (logistic on minutes+hints); chance is 0.5.
- Signal comes from session behaviour (minutes, hints). `prior_score`, `enrol_year` and `school_code` add little.
- The boosted model is no better than (here, slightly worse than) the two-feature logistic, so there is no sign of useful non-linear structure in these features.
- Overall, session-level features give only weak prediction: better than chance, but far from reliable for individual sessions.
- A student's other-session pass rate lifts AUC much higher (section 3), so a large student-level effect is the dominant driver, but it needs prior outcomes and is not available for new students.
- Not shown: causality, and the generalisation to future sessions (no timestamps, so splits are by student, not time).''')